# Async client patterns for your Fantasy Football API

This notebook focuses on a few high-value examples instead of covering every endpoint:

1. **Sync baseline vs `asyncio` vs AnyIO** for paginated `/v0/players/`
2. **Bounded concurrency** for many `/v0/players/{player_id}/` detail calls
3. **Incremental sync** across a few endpoints using `minimum_last_changed_date`

The point is to help you see:
- what async changes
- what AnyIO changes beyond raw `asyncio`
- which patterns are worth using in real client code

## Setup

Install these if needed:

```bash
pip install httpx anyio pandas
```

Start your FastAPI app separately, then set `BASE_URL` below.

In [1]:
from __future__ import annotations

import asyncio
import anyio
import httpx
import time
from datetime import date
from typing import Any, Iterable

BASE_URL = "http://0.0.0.0:8000"
PLAYERS_PAGE_SIZE = 100

# Change this if you want to try a different cutoff for incremental sync.
MINIMUM_LAST_CHANGED_DATE = "2024-01-01"

## A few tiny helpers

In [2]:
def flatten(list_of_lists: Iterable[list[dict[str, Any]]]) -> list[dict[str, Any]]:
    return [item for sublist in list_of_lists for item in sublist]


def summarize_timing(label: str, started: float, result: Any) -> Any:
    elapsed = time.perf_counter() - started
    try:
        size = len(result)
    except TypeError:
        size = "n/a"
    print(f"{label}: {elapsed:.3f}s | size={size}")
    return result

## Example 1: paginated `/v0/players/`

This is the cleanest first comparison.

We'll fetch a few pages of players three ways:
- sync baseline
- `asyncio.gather(...)`
- AnyIO task group

Try increasing `num_pages` to make the differences more obvious.

In [11]:
def get_player_pages_sync(
    base_url: str = BASE_URL,
    num_pages: int = 20,
    page_size: int = PLAYERS_PAGE_SIZE,
) -> list[dict[str, Any]]:
    players: list[dict[str, Any]] = []
    with httpx.Client(base_url=base_url, timeout=30.0) as client:
        for page_num in range(num_pages):
            skip = page_num * page_size
            response = client.get("/v0/players/", params={"skip": skip, "limit": page_size})
            response.raise_for_status()
            players.extend(response.json())
    return players


async def _fetch_player_page_asyncio(
    client: httpx.AsyncClient,
    *,
    skip: int,
    page_size: int,
) -> list[dict[str, Any]]:
    response = await client.get("/v0/players/", params={"skip": skip, "limit": page_size})
    response.raise_for_status()
    return response.json()


async def get_player_pages_asyncio(
    base_url: str = BASE_URL,
    num_pages: int = 5,
    page_size: int = PLAYERS_PAGE_SIZE,
) -> list[dict[str, Any]]:
    async with httpx.AsyncClient(base_url=base_url, timeout=30.0) as client:
        tasks = [
            _fetch_player_page_asyncio(client, skip=page_num * page_size, page_size=page_size)
            for page_num in range(num_pages)
        ]
        pages = await asyncio.gather(*tasks)
    return flatten(pages)


async def _fetch_player_page_anyio(
    client: httpx.AsyncClient,
    skip: int,
    page_size: int,
    results: dict[int, list[dict[str, Any]]],
) -> None:
    response = await client.get("/v0/players/", params={"skip": skip, "limit": page_size})
    response.raise_for_status()
    results[skip] = response.json()


async def get_player_pages_anyio(
    base_url: str = BASE_URL,
    num_pages: int = 20,
    page_size: int = PLAYERS_PAGE_SIZE,
) -> list[dict[str, Any]]:
    results: dict[int, list[dict[str, Any]]] = {}
    async with httpx.AsyncClient(base_url=base_url, timeout=30.0) as client:
        async with anyio.create_task_group() as tg:
            for page_num in range(num_pages):
                skip = page_num * page_size
                tg.start_soon(_fetch_player_page_anyio, client, skip, page_size, results)

    return flatten(results[skip] for skip in sorted(results))

In [12]:
NUM_PAGES = 20

started = time.perf_counter()
players_sync = summarize_timing(
    "sync players",
    started,
    get_player_pages_sync(num_pages=NUM_PAGES),
)

started = time.perf_counter()
players_asyncio = summarize_timing(
    "asyncio players",
    started,
    await get_player_pages_asyncio(num_pages=NUM_PAGES),
)

started = time.perf_counter()
players_anyio = summarize_timing(
    "anyio players",
    started,
    await get_player_pages_anyio(num_pages=NUM_PAGES),
)

print("same count:", len(players_sync) == len(players_asyncio) == len(players_anyio))
print("first 5 ids sync:", [p["player_id"] for p in players_sync[:5]])
print("first 5 ids asyncio:", [p["player_id"] for p in players_asyncio[:5]])
print("first 5 ids anyio:", [p["player_id"] for p in players_anyio[:5]])

sync players: 1.150s | size=1018
asyncio players: 0.677s | size=1018
anyio players: 0.641s | size=1018
same count: True
first 5 ids sync: [1001, 1002, 1003, 1004, 1005]
first 5 ids asyncio: [1001, 1002, 1003, 1004, 1005]
first 5 ids anyio: [1001, 1002, 1003, 1004, 1005]


### What to notice

- The **sync** version is the simplest baseline.
- The **`asyncio`** version is compact and great for straightforward fan-out.
- The **AnyIO** version is a bit more explicit, but the task-group structure scales better once you add cancellation, retries, queues, or more complex workflows.

## Example 2: bounded concurrency for many player detail calls

This is one of the most valuable patterns to learn.

Why it matters:
- detail requests are independent
- async lets you overlap network wait time
- a concurrency limit keeps you from overwhelming your API or client resources

We'll:
1. get a batch of player IDs from `/v0/players/`
2. fetch player detail records from `/v0/players/{player_id}/`
3. compare plain `asyncio` and AnyIO versions with a concurrency limit

In [15]:
def get_some_player_ids_sync(
    base_url: str = BASE_URL,
    *,
    limit: int = 25,
) -> list[int]:
    with httpx.Client(base_url=base_url, timeout=30.0) as client:
        response = client.get("/v0/players/", params={"skip": 0, "limit": limit})
        response.raise_for_status()
        players = response.json()
    return [player["player_id"] for player in players]


PLAYER_IDS = get_some_player_ids_sync(limit=25)
print(PLAYER_IDS[:10])

[1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010]


In [16]:
async def fetch_player_detail_asyncio(
    client: httpx.AsyncClient,
    player_id: int,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    async with semaphore:
        response = await client.get(f"/v0/players/{player_id}/")
        response.raise_for_status()
        return response.json()


async def get_player_details_asyncio(
    player_ids: list[int],
    *,
    base_url: str = BASE_URL,
    concurrency: int = 10,
) -> list[dict[str, Any]]:
    semaphore = asyncio.Semaphore(concurrency)
    async with httpx.AsyncClient(base_url=base_url, timeout=30.0) as client:
        tasks = [fetch_player_detail_asyncio(client, pid, semaphore) for pid in player_ids]
        return await asyncio.gather(*tasks)

In [17]:
async def fetch_player_detail_anyio(
    client: httpx.AsyncClient,
    player_id: int,
    limiter: anyio.CapacityLimiter,
    results: dict[int, dict[str, Any]],
) -> None:
    async with limiter:
        response = await client.get(f"/v0/players/{player_id}/")
        response.raise_for_status()
        results[player_id] = response.json()


async def get_player_details_anyio(
    player_ids: list[int],
    *,
    base_url: str = BASE_URL,
    concurrency: int = 10,
) -> list[dict[str, Any]]:
    limiter = anyio.CapacityLimiter(total_tokens=concurrency)
    results: dict[int, dict[str, Any]] = {}

    async with httpx.AsyncClient(base_url=base_url, timeout=30.0) as client:
        async with anyio.create_task_group() as tg:
            for player_id in player_ids:
                tg.start_soon(fetch_player_detail_anyio, client, player_id, limiter, results)

    return [results[player_id] for player_id in player_ids]

In [18]:
CONCURRENCY = 10

started = time.perf_counter()
details_asyncio = summarize_timing(
    f"asyncio player details (concurrency={CONCURRENCY})",
    started,
    await get_player_details_asyncio(PLAYER_IDS, concurrency=CONCURRENCY),
)

started = time.perf_counter()
details_anyio = summarize_timing(
    f"anyio player details (concurrency={CONCURRENCY})",
    started,
    await get_player_details_anyio(PLAYER_IDS, concurrency=CONCURRENCY),
)

print("same count:", len(details_asyncio) == len(details_anyio))
print("sample names asyncio:", [(p["first_name"], p["last_name"]) for p in details_asyncio[:5]])
print("sample names anyio:", [(p["first_name"], p["last_name"]) for p in details_anyio[:5]])

asyncio player details (concurrency=10): 0.224s | size=25
anyio player details (concurrency=10): 0.200s | size=25
same count: True
sample names asyncio: [('Aaron', 'Rodgers'), ('Matt', 'Prater'), ('Marcedes', 'Lewis'), ('Nick', 'Folk'), ('Mason', 'Crosby')]
sample names anyio: [('Aaron', 'Rodgers'), ('Matt', 'Prater'), ('Marcedes', 'Lewis'), ('Nick', 'Folk'), ('Mason', 'Crosby')]


### What to notice

This is the pattern you'll probably reuse the most in real client code.

The main design idea is not just “use async.” It is:

- use **async**
- but keep **bounded concurrency**
- and keep your request ordering and result collection explicit

## Example 3: incremental sync across a few endpoints

Your API supports `minimum_last_changed_date` on several collection endpoints.  
That makes it natural to do incremental pulls instead of full refreshes.

We'll fetch changed records from:
- `/v0/players/`
- `/v0/teams/`
- `/v0/leagues/`
- `/v0/performances/`

This example is especially useful for data engineering / notebook workflows.

In [19]:
ENDPOINTS = [
    ("players", "/v0/players/"),
    ("teams", "/v0/teams/"),
    ("leagues", "/v0/leagues/"),
    ("performances", "/v0/performances/"),
]

In [20]:
async def incremental_sync_asyncio(
    minimum_last_changed_date: str = MINIMUM_LAST_CHANGED_DATE,
    *,
    base_url: str = BASE_URL,
    page_size: int = 100,
) -> dict[str, list[dict[str, Any]]]:
    async with httpx.AsyncClient(base_url=base_url, timeout=30.0) as client:
        async def fetch_endpoint(name: str, path: str) -> tuple[str, list[dict[str, Any]]]:
            response = await client.get(
                path,
                params={
                    "skip": 0,
                    "limit": page_size,
                    "minimum_last_changed_date": minimum_last_changed_date,
                },
            )
            response.raise_for_status()
            return name, response.json()

        tasks = [fetch_endpoint(name, path) for name, path in ENDPOINTS]
        pairs = await asyncio.gather(*tasks)
        return dict(pairs)

In [21]:
async def incremental_sync_anyio(
    minimum_last_changed_date: str = MINIMUM_LAST_CHANGED_DATE,
    *,
    base_url: str = BASE_URL,
    page_size: int = 100,
) -> dict[str, list[dict[str, Any]]]:
    results: dict[str, list[dict[str, Any]]] = {}

    async def fetch_endpoint(
        client: httpx.AsyncClient,
        name: str,
        path: str,
    ) -> None:
        response = await client.get(
            path,
            params={
                "skip": 0,
                "limit": page_size,
                "minimum_last_changed_date": minimum_last_changed_date,
            },
        )
        response.raise_for_status()
        results[name] = response.json()

    async with httpx.AsyncClient(base_url=base_url, timeout=30.0) as client:
        async with anyio.create_task_group() as tg:
            for name, path in ENDPOINTS:
                tg.start_soon(fetch_endpoint, client, name, path)

    return results

In [23]:
started = time.perf_counter()
changed_asyncio = summarize_timing(
    "asyncio incremental sync",
    started,
    await incremental_sync_asyncio(),
)

started = time.perf_counter()
changed_anyio = summarize_timing(
    "anyio incremental sync",
    started,
    await incremental_sync_anyio(),
)

print({k: len(v) for k, v in changed_asyncio.items()})
print({k: len(v) for k, v in changed_anyio.items()})

asyncio incremental sync: 0.112s | size=4
anyio incremental sync: 0.138s | size=4
{'players': 100, 'teams': 20, 'leagues': 5, 'performances': 100}
{'leagues': 5, 'performances': 100, 'teams': 20, 'players': 100}


## Optional: turn results into DataFrames

This is handy in notebooks because it lets you move from async ingestion into normal data analysis.

In [24]:
import pandas as pd

players_df = pd.DataFrame(players_anyio)
details_df = pd.DataFrame(details_anyio)
performances_df = pd.DataFrame(changed_anyio.get("performances", []))

display(players_df.head())
display(details_df[["player_id", "first_name", "last_name", "position"]].head())
display(performances_df.head())

,player_id,gsis_id,first_name,last_name,position,last_changed_date,performances
0,1001,00-0023459,Aaron,Rodgers,QB,2024-04-18,"[{'performance_id': 2501, 'player_id': 1001, '..."
1,1002,00-0023853,Matt,Prater,K,2024-04-18,"[{'performance_id': 2502, 'player_id': 1002, '..."
2,1003,00-0024243,Marcedes,Lewis,TE,2024-04-18,"[{'performance_id': 2503, 'player_id': 1003, '..."
3,1004,00-0025565,Nick,Folk,K,2024-04-18,"[{'performance_id': 2504, 'player_id': 1004, '..."
4,1005,00-0025580,Mason,Crosby,K,2024-04-18,"[{'performance_id': 2505, 'player_id': 1005, '..."


,player_id,first_name,last_name,position
0,1001,Aaron,Rodgers,QB
1,1002,Matt,Prater,K
2,1003,Marcedes,Lewis,TE
3,1004,Nick,Folk,K
4,1005,Mason,Crosby,K


,performance_id,player_id,week_number,fantasy_points,last_changed_date
0,2501,1001,202301,20.0,2024-03-01
1,2502,1002,202301,22.0,2024-03-01
2,2503,1003,202301,13.0,2024-03-01
3,2504,1004,202301,15.0,2024-03-01
4,2505,1005,202301,3.0,2024-03-01


## Suggested experiments

Try these changes and rerun:

1. Increase `NUM_PAGES` from 5 to 20
2. Change `CONCURRENCY` from 5 to 10 to 25
3. Increase the number of player IDs in the detail example
4. Change `MINIMUM_LAST_CHANGED_DATE`
5. Add a small sleep in your API to exaggerate the difference between sync and async

The most important thing to observe is not just raw speed. Also notice:
- code shape
- result ordering
- where concurrency limits live
- which style feels easier to reason about

## Takeaways

For this API, the highest-value async client patterns are:

- **paginated fan-out**
- **bounded concurrency for detail calls**
- **incremental sync across multiple endpoints**

And the best reason to learn AnyIO here is not that it makes everything magically faster.  
It gives you a cleaner model for **task groups, cancellation, concurrency control, and more complex async workflows** as your client code grows.